# M9.17 · Video ve hareket — blok eşleme, optik akış, FRUC, stabilizasyon

Videoyu zaman ekseniyle birlikte işlemek: hareket tespiti ve hareket kestirimi, açıklık (aperture), örtme ve paralaks sorunları; blok eşleme (SAD) ve hareket telafili kestirim, Lucas–Kanade optik akışı ve piramit sürümü, ileri–geri tutarlılık; zamansal gürültü giderme, kare hızı artırma (FRUC) ve video stabilizasyonu. Saf Python'da tam arama blok eşleme; NumPy'da tüm blokları ve kaydırmaları tek seferde arayan blok eşleme ve tek seviyeli LK; PyTorch'ta piramitli yoğun LK, ileri–geri kontrol ve hareket telafili ara kare üretimi; en sonda çevrimdışı stabilizasyonu kamerayı yumuşatıp tüm kareleri tek `grid_sample` çağrısıyla düzelterek yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-17-motion.html](https://m-gonul.github.io/pytorch-lab/m9-17-motion.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_17_motion.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import no_imports, need, ensure\n\n\ndef ref_bm(prev, cur, bs, R):\n    H, W = cur.shape\n    out = np.zeros((H // bs, W // bs, 2), int)\n    for i in range(H // bs):\n        for j in range(W // bs):\n            c = cur[i * bs:(i + 1) * bs, j * bs:(j + 1) * bs].astype(int)\n            best = None\n            for dy in range(-R, R + 1):\n                for dx in range(-R, R + 1):\n                    y, x = i * bs + dy, j * bs + dx\n                    if 0 <= y <= H - bs and 0 <= x <= W - bs:\n                        s = np.abs(c - prev[y:y + bs, x:x + bs]).sum()\n                        if best is None or s < best[0]:\n                            best = (s, dy, dx)\n            out[i, j] = best[1:]\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/cv2 kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"scipy\", \"skimage\"}))\n    prev = data.camera()[200:248, 200:256].astype(int)\n    M = np.float32([[1, 0, 2], [0, 1, -1]])\n    cur = cv2.warpAffine(data.camera(), M, (512, 512))[200:248, 200:256].astype(int)\n\n    def bm():\n        got = np.array(need(ns, \"block_match\")(prev.tolist(), cur.tolist(), 8, 3))\n        want = ref_bm(prev, cur, 8, 3)\n        ensure(got.shape == want.shape and np.array_equal(got, want), \"başvuru ile farklı\")\n        inner = got[1:-1, 1:-1].reshape(-1, 2)\n        ensure((inner == [1, -2]).all(1).mean() > 0.8, \"iç blokların çoğu (dy, dx) = (1, −2) olmalı\")\n    R.test(\"block_match (tam arama)\", bm)\n\n    def mc():\n        mv = need(ns, \"block_match\")(prev.tolist(), cur.tolist(), 8, 3)\n        pred = np.array(need(ns, \"compensate\")(prev.tolist(), mv, 8))\n        P = lambda a: 10 * np.log10(255 ** 2 / ((a - cur) ** 2).mean())\n        ensure(P(pred) > P(prev) + 5, \"telafi, önceki kareyi kopyalamaktan çok daha iyi olmalı\")\n        return f\"önceki kare PSNR {P(prev):.1f} dB → hareket telafili {P(pred):.1f} dB\"\n    R.test(\"compensate\", mc)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef ref_bm(prev, cur, bs, R):\n    H, W = cur.shape\n    out = np.zeros((H // bs, W // bs, 2), int)\n    for i in range(H // bs):\n        for j in range(W // bs):\n            c = cur[i * bs:(i + 1) * bs, j * bs:(j + 1) * bs].astype(int)\n            best = None\n            for dy in range(-R, R + 1):\n                for dx in range(-R, R + 1):\n                    y, x = i * bs + dy, j * bs + dx\n                    if 0 <= y <= H - bs and 0 <= x <= W - bs:\n                        s = np.abs(c - prev[y:y + bs, x:x + bs]).sum()\n                        if best is None or s < best[0]:\n                            best = (s, dy, dx)\n            out[i, j] = best[1:]\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n\n    def bm():\n        prev = img[100:228, 150:278]\n        cur = cv2.warpAffine(img, np.float32([[1, 0, -3], [0, 1, 2]]), (512, 512))[100:228, 150:278]\n        got = need(ns, \"block_match_np\")(prev, cur, 8, 5)\n        ensure(np.array_equal(got, ref_bm(prev.astype(int), cur.astype(int), 8, 5)), \"başvuru (döngülü tam arama) ile farklı\")\n    R.test(\"block_match_np = tam arama\", bm)\n\n    def lk():\n        I0 = cv2.GaussianBlur(img.astype(np.float64), (0, 0), 1.5)\n        rows = []\n        for u, v in [(0.3, -0.2), (0.6, 0.4)]:\n            I1 = cv2.warpAffine(I0, np.float64([[1, 0, u], [0, 1, v]]), (512, 512), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)\n            uu, vv = need(ns, \"lk_flow\")(I0, I1)\n            Iy, Ix = np.gradient(I0)\n            from scipy import ndimage\n            f = lambda a: ndimage.uniform_filter(a, 7, mode=\"nearest\")\n            a, b, c = f(Ix * Ix), f(Ix * Iy), f(Iy * Iy)\n            lmin = (a + c) / 2 - np.sqrt(((a - c) / 2) ** 2 + b * b)\n            m = lmin > 20; m[:16] = m[-16:] = False; m[:, :16] = m[:, -16:] = False\n            e = np.hypot(uu - u, vv - v)[m]\n            rows.append([f\"({u}, {v})\", f\"{np.median(uu[m]):.3f}, {np.median(vv[m]):.3f}\", f\"{np.median(e):.3f}\"])\n            ensure(np.median(e) < 0.15, f\"medyan uç nokta hatası {np.median(e):.3f}\")\n        R.table(\"Tek adımlı LK, dokulu pikseller\", [\"gerçek (u, v)\", \"medyan kestirim\", \"medyan hata\"], rows,\n                note=\"Tek adım küçük kaymalarda iyidir; büyük kaymalarda doğrusallaştırma bozulur → iterasyon ve piramit (d).\")\n    R.test(\"lk_flow\", lk)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef textured(I0, thr=20):\n    Iy, Ix = np.gradient(I0)\n    from scipy import ndimage\n    f = lambda a: ndimage.uniform_filter(a, 7, mode=\"nearest\")\n    a, b, c = f(Ix * Ix), f(Ix * Iy), f(Iy * Iy)\n    m = (a + c) / 2 - np.sqrt(((a - c) / 2) ** 2 + b * b) > thr\n    m[:40] = m[-40:] = False; m[:, :40] = m[:, -40:] = False\n    return m\n\n\ndef check(ns, R):\n    I0n = cv2.GaussianBlur(data.camera().astype(np.float64), (0, 0), 1.0)\n    I0 = torch.tensor(I0n)\n    m = textured(I0n)\n\n    def wp():\n        f = torch.zeros(2, 512, 512, dtype=torch.float64); f[0] = 3; f[1] = -2\n        close(need(ns, \"warp\")(I0, f)[50:-50, 50:-50].numpy(), I0n[48:-52, 53:-47], atol=1e-9, what=\"tamsayı kayma\")\n    R.test(\"warp\", wp)\n\n    def pl():\n        rows = []\n        for u, v in [(0.4, -0.3), (3.2, -1.7), (9.5, 6.0)]:\n            I1 = cv2.warpAffine(I0n, np.float64([[1, 0, u], [0, 1, v]]), (512, 512), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)\n            fl = need(ns, \"pyr_lk\")(I0, torch.tensor(I1)).numpy()\n            e = np.hypot(fl[0] - u, fl[1] - v)[m]\n            one = ns[\"lk_level\"](I0, torch.tensor(I1), torch.zeros(2, 512, 512, dtype=torch.float64)).numpy()\n            e1 = np.hypot(one[0] - u, one[1] - v)[m]\n            rows.append([f\"({u}, {v})\", f\"{np.median(e1):.3f}\", f\"{np.median(e):.3f}\", f\"%{(e < 0.5).mean() * 100:.0f}\"])\n            ensure(np.median(e) < 0.1 and (e < 0.5).mean() > 0.95, f\"({u}, {v}): medyan hata {np.median(e):.3f}\")\n        R.table(\"Yoğun LK, dokulu pikseller\", [\"gerçek akış\", \"tek seviye medyan hata\", \"4 seviye medyan hata\", \"< 0.5 px\"], rows)\n    R.test(\"pyr_lk büyük hareketi yakalar\", pl)\n\n    def fb():\n        I1 = torch.tensor(cv2.warpAffine(I0n, np.float64([[1, 0, 4.3], [0, 1, 2.1]]), (512, 512), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT))\n        f01, f10 = ns[\"pyr_lk\"](I0, I1), ns[\"pyr_lk\"](I1, I0)\n        e = need(ns, \"fb_error\")(f01, f10).numpy()[m]\n        ensure((e < 0.5).mean() > 0.9, f\"tutarlı piksel oranı {(e < 0.5).mean():.2f}\")\n        return f\"ileri–geri hata < 0.5 px: %{(e < 0.5).mean() * 100:.0f} (dokulu pikseller)\"\n    R.test(\"fb_error\", fb)\n\n    def fr():\n        bg = I0n * 0.6\n        obj = cv2.GaussianBlur(data.coins()[100:180, 100:180].astype(np.float64), (0, 0), 1.0)\n        frames = []\n        for t in range(3):\n            f = bg.copy(); x0 = 150 + 6 * t; y0 = 200 + 3 * t\n            f[y0:y0 + 80, x0:x0 + 80] = obj\n            frames.append(f)\n        A, B, C = (torch.tensor(f) for f in frames)\n        fl = ns[\"pyr_lk\"](A, C)\n        mc = need(ns, \"fruc_mc\")(A, C, fl).numpy()\n        av = (frames[0] + frames[2]) / 2\n        reg = (slice(190, 300), slice(140, 260))\n        P = lambda x: 10 * np.log10(255 ** 2 / ((x[reg] - frames[1][reg]) ** 2).mean())\n        ensure(P(mc) > P(av) + 6, f\"MC {P(mc):.1f} dB, ortalama {P(av):.1f} dB\")\n        R.table(\"Ara kare (nesne 12 px kayıyor), hareketli bölge\", [\"yöntem\", \"PSNR (dB)\"], [[\"kare tekrarı\", f\"{P(frames[0]):.1f}\"], [\"ortalama (hayalet)\", f\"{P(av):.1f}\"], [\"LK + hareket telafisi\", f\"{P(mc):.1f}\"]])\n    R.test(\"fruc_mc\", fr)\n",
 "e": "import math\nimport numpy as np\nimport cv2\nimport torch\nfrom skimage import data, color\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef shaky(T=40, seed=0):\n    rng = np.random.default_rng(seed)\n    big = (color.rgb2gray(data.astronaut()) * 255).astype(np.float32)\n    frames, true = [], []\n    for t in range(T):\n        sx, sy, sa = 2.0 * t, 10 * math.sin(t / 10), 0.0\n        jx, jy, ja = rng.normal(0, 3), rng.normal(0, 3), rng.normal(0, math.radians(0.6))\n        x, y, a = 80 + sx + jx, 120 + sy + jy, sa + ja\n        M = cv2.getRotationMatrix2D((x + 160, y + 120), math.degrees(a), 1.0)\n        M[:, 2] -= [x, y]\n        frames.append(cv2.warpAffine(big, M, (320, 240), flags=cv2.INTER_LINEAR))\n        true.append((jx, jy, ja))\n    return np.stack(frames), np.array(true)\n\n\ndef jitter(frames, est):\n    m = est(frames)\n    traj = np.vstack([[0, 0, 0], np.cumsum(m, 0)])\n    k = 5\n    P = np.pad(traj, ((k, k), (0, 0)), mode=\"edge\")\n    sm = np.stack([P[i:i + 2 * k + 1].mean(0) for i in range(len(traj))])\n    return np.abs(traj - sm)[k:-k, :2].mean()\n\n\ndef check(ns, R):\n    frames, _ = shaky()\n\n    def sm():\n        traj = np.random.default_rng(1).normal(size=(30, 3)).cumsum(0)\n        got = need(ns, \"smooth\")(traj, 4)\n        P = np.pad(traj, ((4, 4), (0, 0)), mode=\"edge\")\n        close(got, np.stack([P[i:i + 9].mean(0) for i in range(30)]), atol=1e-12, what=\"hareketli ortalama\")\n    R.test(\"smooth (cumsum)\", sm)\n\n    def em():\n        m = need(ns, \"estimate_motion\")(frames[:6])\n        ensure(m.shape == (5, 3), \"(T−1, 3)\")\n    R.test(\"estimate_motion\", em)\n\n    def st():\n        out, corr = need(ns, \"stabilize\")(frames)\n        ensure(out.shape == frames.shape, \"şekil\")\n        crop = lambda f: f[:, 30:-30, 40:-40]\n        j0 = jitter(crop(frames), ns[\"estimate_motion\"])\n        j1 = jitter(crop(out), ns[\"estimate_motion\"])\n        ensure(j1 < 0.4 * j0, f\"titreşim {j0:.2f} → {j1:.2f} piksel\")\n        t1 = bench(lambda: ns[\"stabilize\"](frames), repeat=1, number=1)\n        H, W = frames.shape[1:]\n        t2 = bench(lambda: [cv2.warpAffine(f, np.float32([[1, 0, 1], [0, 1, 1]]), (W, H)) for f in frames], repeat=3)\n        R.table(\"40 kare, 320 × 240\", [\"ölçü\", \"değer\"], [[\"girdi titreşimi (yörüngenin yumuşaktan sapması, px)\", f\"{j0:.2f}\"], [\"stabilize çıktı\", f\"{j1:.2f}\"],\n                                                         [\"stabilize (kestirim + tek grid_sample)\", fmt_time(t1)], [\"yalnız 40 cv2.warpAffine\", fmt_time(t2)]])\n    R.test(\"stabilize\", st)\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef video(T=9, sigma=12, seed=0):\n    rng = np.random.default_rng(seed)\n    bg = data.camera()[100:228, 100:228].astype(np.float64)\n    obj = data.coins()[50:90, 50:90].astype(np.float64)\n    clean = []\n    for t in range(T):\n        f = bg.copy(); f[40:80, 10 + 8 * t:50 + 8 * t] = obj\n        clean.append(f)\n    clean = np.stack(clean)\n    return clean, clean + rng.normal(0, sigma, clean.shape)\n\n\ndef check(ns, R):\n    clean, noisy = video()\n\n    def tm():\n        got = need(ns, \"temporal_mean\")(noisy, 2)\n        close(got[4], noisy[2:7].mean(0), what=\"orta kare\"); close(got[0], noisy[0:3].mean(0), what=\"uç kare\")\n    R.test(\"temporal_mean\", tm)\n\n    def cmp():\n        P = lambda a, reg=(slice(None), slice(None)): np.mean([10 * np.log10(255 ** 2 / ((a[t][reg] - clean[t][reg]) ** 2).mean()) for t in range(2, 7)])\n        mov = (slice(40, 80), slice(None))\n        sta = (slice(90, 128), slice(None))\n        rows = [[n, f\"{P(a):.2f}\", f\"{P(a, mov):.2f}\", f\"{P(a, sta):.2f}\"] for n, a in [(\"gürültülü\", noisy), (\"uzaysal Gauss\", need(ns, \"spatial\")(noisy)), (\"zamansal ortalama (5 kare)\", ns[\"temporal_mean\"](noisy, 2)), (\"hareket kapılı ortalama\", need(ns, \"gated_mean\")(noisy, 2, 12))]]\n        d = {r[0]: (float(r[1]), float(r[2]), float(r[3])) for r in rows}\n        ensure(d[\"hareket kapılı ortalama\"][1] > d[\"zamansal ortalama (5 kare)\"][1] + 2, \"kapı hareketli bölgede hayaleti önlemeli\")\n        ensure(d[\"zamansal ortalama (5 kare)\"][2] > d[\"gürültülü\"][2] + 5, \"zamansal ortalama durağan bölgede PSNR'ı belirgin artırmalı\")\n        R.table(\"σ = 12 gürültü, nesne kare başına 8 px kayıyor\", [\"yöntem\", \"PSNR (tüm kare)\", \"PSNR (hareketli şerit)\", \"PSNR (durağan bölge)\"], rows,\n                note=\"Düz zamansal ortalama durağan bölgede √5 kat gürültü azaltır ama hareketli nesneyi hayalete çevirir; kapı ikisini birden yakalar.\")\n    R.test(\"karşılaştırma\", cmp)\n",
 "x2": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    I0 = cv2.GaussianBlur(data.camera()[128:384, 128:384].astype(np.float64), (0, 0), 1.5)\n\n    def t():\n        H, W = I0.shape\n        M = cv2.getRotationMatrix2D((W / 2, H / 2), 1.0, 1.0)\n        I1 = cv2.warpAffine(I0, M, (W, H), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)\n        ys, xs = np.mgrid[:H, :W].astype(float)\n        p = np.stack([xs, ys, np.ones_like(xs)], -1) @ M.T\n        gu, gv = p[..., 0] - xs, p[..., 1] - ys\n        u, v = need(ns, \"horn_schunck\")(I0, I1, 10.0, 300)\n        inner = (slice(30, -30), slice(30, -30))\n        e = np.hypot(u - gu, v - gv)[inner]\n        Iy, Ix = np.gradient(I0)\n        from scipy import ndimage\n        f = lambda a: ndimage.uniform_filter(a, 7, mode=\"nearest\")\n        a, b, c = f(Ix * Ix), f(Ix * Iy), f(Iy * Iy); p_, q_ = f(Ix * (I1 - I0)), f(Iy * (I1 - I0))\n        det = a * c - b * b\n        lk_u, lk_v = np.where(det > 1e-3, (-c * p_ + b * q_) / np.where(det > 1e-3, det, 1), 0), np.where(det > 1e-3, (b * p_ - a * q_) / np.where(det > 1e-3, det, 1), 0)\n        el = np.hypot(lk_u - gu, lk_v - gv)[inner]\n        ensure(np.mean(e) < np.mean(el), f\"HS ortalama hata {np.mean(e):.3f}, LK {np.mean(el):.3f}\")\n        R.table(\"1° dönme (akış merkezden uzaklıkla büyür, en çok ≈ 2.2 px)\", [\"yöntem\", \"ortalama EPE\", \"%90'lık EPE\"],\n                [[\"LK (7 × 7, tek adım)\", f\"{np.mean(el):.3f}\", f\"{np.percentile(el, 90):.3f}\"], [\"Horn–Schunck (α = 10, 300 it.)\", f\"{np.mean(e):.3f}\", f\"{np.percentile(e, 90):.3f}\"]],\n                note=\"LK düz bölgelerde akış veremez (0 kalır); HS düzgünlük terimiyle akışı dokusuz bölgelere yayar.\")\n    R.test(\"Horn–Schunck vs LK\", t)\n",
 "x3": "import numpy as np\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    bg = data.camera()[200:296, 150:278].astype(np.float64) * 0.7\n    obj = data.coins()[60:100, 60:100].astype(np.float64)\n    fr = []\n    for t in range(3):\n        f = bg.copy(); f[28:68, 20 + 6 * t:60 + 6 * t] = obj\n        fr.append(f)\n\n    def t():\n        P = lambda a: 10 * np.log10(255 ** 2 / ((a - fr[1]) ** 2).mean())\n        rows = [[n, f\"{P(need(ns, fn)(fr[0], fr[2])):.2f}\"] for n, fn in [(\"tekrar\", \"fruc_repeat\"), (\"ortalama\", \"fruc_average\"), (\"simetrik blok eşleme\", \"fruc_block\")]]\n        v = [float(r[1]) for r in rows]\n        ensure(v[2] > v[1] + 5 and v[2] > v[0] + 5, f\"blok FRUC en iyi olmalı: {v}\")\n        R.table(\"Ara kare (nesne 12 px kayıyor)\", [\"yöntem\", \"PSNR (dB)\"], rows,\n                note=\"Simetrik arama ara karenin ızgarasında çalıştığı için delik ya da çakışma oluşmaz; gerçek videoda blok sınırlarında artefakt kalır (OBMC ya da yoğun akış çözümü).\")\n    R.test(\"FRUC karşılaştırması\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_17_motion as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Video** zamanla değişen bir görüntüdür: $f(x,y,t)$. Tek kareli yöntemler kareleri bağımsız işler; çok kareli yöntemler komşu kareleri de kullanır ve hareket bilgisi sayesinde genelde çok daha iyidir. Hareket iki biçimde kullanılır: **tespit** (bu piksel hareket ediyor mu? ucuz) ve **kestirim** (hareket vektörü ne? pahalı ama "video işlemenin kutsal kâsesi").

**Kötü konumlanmışlık:** 2-B hareket 3-B hareketin izdüşümüdür ve tek başına belirlenemez: **açıklık** (düz kenarda yalnız dik bileşen), **örtme** (bir karede görünen bölge diğerinde yok), **paralaks** (derinliğe bağlı farklı hareket). Düzenlileştirme gerekir.

**Blok eşleme:** kareyi bloklara böl, her blok için önceki karede $\pm R$ arama penceresinde maliyeti (SAD, MAD, SSD, MSE) en küçük kaydırmayı bul. Tüm video kodlayıcılarının kalbidir; hareket telafili kestirimde blok, bulunan vektörle önceki kareden kopyalanır.

**Optik akış:** parlaklık sabitliği $I(x+u,y+v,t+1)=I(x,y,t)$ → $I_xu+I_yv+I_t=0$: bir denklem, iki bilinmeyen. **Lucas–Kanade:** akış küçük bir pencerede sabit → $A\mathbf d=-\mathbf b$, $A=[I_x\ I_y]$, en küçük kareler $\begin{bmatrix}\sum I_x^2&\sum I_xI_y\\\sum I_xI_y&\sum I_y^2\end{bmatrix}\begin{bmatrix}u\\v\end{bmatrix}=-\begin{bmatrix}\sum I_xI_t\\\sum I_yI_t\end{bmatrix}$. $A^TA$ iyi koşullu olmalı (Harris matrisi! M9.7). Büyük hareket için **piramit** (Bouguet): kabadan inceye, her seviyede akışı ikiyle çarpıp başlangıç al. **Horn–Schunck:** tüm görüntüde $\iint(I_xu+I_yv+I_t)^2+\alpha^2(|\nabla u|^2+|\nabla v|^2)$. Güvenilirlik: **ileri–geri tutarlılık** $f_{01}(x)+f_{10}(x+f_{01}(x))\approx0$.

**Zamansal gürültü giderme:** aynı konumdaki pikselleri karelerde ortala (hareket varsa hayalet); hareket tespit edilen pikselleri dışarıda bırak; en iyisi hareketi telafi edip benzerliğe göre ağırlıklandır.

**FRUC (kare hızı artırma):** tekrar (yapay ama artefaktsız), ortalama (hayalet), blok hareket telafili, yoğun akış telafili, derin ağlar (RIFE). Telafili FRUC: akış → çarpıtma → birleştirme.

**Stabilizasyon:** (1) kamera yörüngesini kareler arası dönüşümlerden kestir (öznitelik takibi + benzerlik dönüşümü), (2) istenmeyen (yüksek frekanslı) hareketi yörüngeyi yumuşatarak ayır, (3) her kareyi farkla düzelt; kenarlarda kırpma gerekir.

:::ayrıntı Video sabitleme neden zor? Zorluklar ve yöntem aileleri
**Kamera hareketi modeli:** küresel 2B (öteleme, benzerlik, afin, homografi — tek dönüşüm bütün kareye), yerel 2B (karenin bölgelerine ayrı dönüşüm: ağ/mesh tabanlı), 3B (kamera pozu ve sahne derinliği). **Zorluklar:** düşük dokulu arka planda öznitelik bulunamaz; derinlik farkı (paralaks) yüzünden tek bir 2B dönüşüm bütün sahneyi açıklayamaz; hareketli ön plan nesneleri kamera hareketi sanılır; CMOS algılayıcılarda satırlar farklı anlarda okunur (**rolling shutter**): hızlı harekette kare "jöle" gibi eğilir; istenen (kaydırma) ile istenmeyen (titreme) hareketi ayırmak — genelde frekansla: yüksek frekans istenmeyen. **Ödünleşimler:** yumuşaklık ↔ sadakat (yörüngeyi çok yumuşatırsan orijinal kadrajdan uzaklaşırsın, daha çok kırpman gerekir); kırpma ↔ boş kenar (ya tek boyutta kırp ve görüş alanını kaybet ya da kenarları doldur/inpainting yap). **Yöntem aileleri:** süzgeç tabanlı (yörüngeye alçak geçiren süzgeç, ör. Gauss/Kalman; çevrimiçi çalışır), enterpolasyon tabanlı (anahtar karelere spline), optimizasyon tabanlı (yörüngenin 1., 2., 3. türevlerinin L1 normunu kırpma kısıtları altında en aza indir — sabit, sabit hızlı, sabit ivmeli parçalar; çevrimdışı), derin öğrenme tabanlı (ağ, kararsız kareden doğrudan çarpıtma alanı tahmin eder). **Çevrimiçi** (canlı, yalnız geçmiş kareler) ve **çevrimdışı** (tüm video) işleme farklı yöntemler gerektirir.
:::

## 1 · Elle

<!-- çözüm -->

LK: $10u=20$ → $u=2$; $5v=-5$ → $v=-1$.

Tam arama blok başına $225\times256=57\,600$ mutlak fark: 1080p'de (8160 blok) kare başına ~470 milyon işlem — kodlayıcıların hızlı arama sezgiselleri bu yüzden var.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Lucas–Kanade: $\sum I_x^2=10$, $\sum I_xI_y=0$, $\sum I_y^2=5$, $\sum I_xI_t=-20$, $\sum I_yI_t=5$**

- `u`: $u$
- `v`: $v$

**Blok eşleme ve video**

- `cand`: 16 × 16 blok, arama ±7: aday sayısı
- `ops`: Tam aramada blok başına mutlak fark sayısı
- `w0`: Kareler 0 ve 1 arasında $t=0.25$'te ağırlıklı ortalama: kare 0'ın ağırlığı
- `tn`: σ = 10 gürültü, 4 hareketsiz kare ortalaması: kalan σ
- `fb`: İleri (3, −2), geri (−2.6, 2.1): $\lVert f+b\rVert$

In [ ]:
cevaplar = dict(
    u=None,
    v=None,
    cand=None,
    ops=None,
    w0=None,
    tn=None,
    fb=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"u":[2,0],"v":[-1,0],"cand":[225,0],"ops":[57600,0],"w0":[0.75,0],"tn":[5,0],"fb":[0.412311,0.0001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, liste listeleri (gri seviye).

- `block_match(prev, cur, bs, R)` — `cur`'un her `bs × bs` bloğu (satır sırası, artakalan kenar atlanır) için önceki karede `dy, dx ∈ [-R, R]` (dış döngü `dy`, iç `dx`), aday blok tamamen görüntü içindeyse SAD; en küçük SAD (eşitlikte ilk bulunan). `[[(dy, dx), …], …]` blok ızgarası.
- `compensate(prev, mvs, bs)` — her bloğu `prev`'in kaydırılmış bloğundan kopyalayarak kestirim karesi; kapsanmayan kenar pikselleri `prev`'den.
- `sad(a, b)` yardımcısı.

In [ ]:
%%writefile stage_b.py
def sad(a, b):
    ...


def block_match(prev, cur, bs, R):
    ...


def compensate(prev, mvs, bs):
    ...


prev = [[(x * 7 + y * 3) % 50 for x in range(16)] for y in range(16)]
cur = [row[1:] + row[:1] for row in prev]
print(block_match(prev, cur, 8, 2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**.

- `block_match_np(prev, cur, bs, R)` — b ile aynı sonuç, `(nby, nbx, 2)` `int` dizi. `prev`'i `R` ile **büyük bir değerle** (ör. `1e9`'luk `float`) doldurup `sliding_window_view` ile tüm adayların bloklarını, `cur` bloklarını yeniden şekillendir, SAD hacmi `(nby, nbx, (2R+1)², bs, bs)` → toplam → `argmin` (görüntü dışına taşan adaylar dev SAD alır, böylece seçilmez).
- `lk_flow(I0, I1, win=7, thr=1e-2)` — tek seviyeli, tek adımlı LK: $I_x, I_y$ merkezi fark / 2 (`np.gradient`), $I_t=I_1-I_0$, pencere toplamları `scipy.ndimage.uniform_filter(…, win, mode="nearest")` (ortalama da olur, oran değişmez), $\lambda_{min}>$ `thr` olan yerlerde 2 × 2 çözüm, diğer yerlerde 0. `(u, v)`.

In [ ]:
%%writefile stage_c.py
import numpy as np
from scipy import ndimage


def block_match_np(prev, cur, bs, R):
    ...


def lk_flow(I0, I1, win=7, thr=1e-2):
    ...


from skimage import data
p = data.camera()[:64, :64]
print(block_match_np(p, np.roll(p, 1, 1), 8, 2)[2, 2])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Yoğun, piramitli Lucas–Kanade (PyTorch).

- `warp(img, flow)` — `img (H, W)`, `flow (2, H, W)`: $img(x+u, y+v)$, `grid_sample` (`align_corners=True`, `padding_mode="border"`).
- `lk_level(I0, I1, flow, win=7, iters=5)` — $I_0$'ın gradyanları (merkezi fark / 2, `replicate`), pencere ortalamaları (`avg_pool2d`, `replicate` dolgu); her iterasyonda $I_t=$`warp(I1, flow)`$-I_0$, $\lambda_{min}>10^{-2}$ olan yerlerde LK adımı (adım ±1 pikselle kırpılır), ardından akışı **güvenle ağırlıklı** pencere ortalamasıyla düzgünleştir: $f\leftarrow\frac{\text{box}(\lambda_{min}f)}{\text{box}(\lambda_{min})}$ (yoksa düz bölgelerin anlamsız akışları komşu pencerelerin $I_t$'sini bozar).
- `pyr_lk(I0, I1, levels=4, win=7, iters=5)` — `avg_pool2d` ile piramit; kabadan inceye, seviye değişirken `F.interpolate(…, mode="bilinear")` ve ×2.
- `fb_error(f01, f10)` — $\lVert f_{01}(x)+f_{10}(x+f_{01}(x))\rVert$ (`warp` ile).
- `fruc_mc(I0, I2, flow02)` — ara kare $\frac12[I_0(x-\frac12f)+I_2(x+\frac12f)]$ (`warp` ile; akışın ara noktada da aynı olduğu yaklaşımı).

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def warp(img, flow):
    ...


def lk_level(I0, I1, flow, win=7, iters=5):
    ...


def pyr_lk(I0, I1, levels=4, win=7, iters=5):
    ...


def fb_error(f01, f10):
    ...


def fruc_mc(I0, I2, flow02):
    ...


x = torch.rand(32, 32, dtype=torch.float64)
print(warp(x, torch.zeros(2, 32, 32, dtype=torch.float64)).allclose(x))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Çevrimdışı video stabilizasyonu.

- `estimate_motion(frames)` → `(T-1, 3)` `[dx, dy, da]`: ardışık kareler arası benzerlik dönüşümü: `cv2.goodFeaturesToTrack(prev, 300, 0.01, 10)`, `cv2.calcOpticalFlowPyrLK`, `cv2.estimateAffinePartial2D` (RANSAC); `dx, dy = M[:, 2]`, `da = atan2(M[1, 0], M[0, 0])`. (Kare döngüsü serbest.)
- `smooth(traj, r=5)` — `(T, 3)` yörüngenin `2r+1`'lik hareketli ortalaması, uçlar `edge` dolgulu, **`cumsum` ile döngüsüz**.
- `stabilize(frames, r=5)` → `(out, corr)`: yörünge = `cumsum` (ilk satır 0), düzeltme = yumuşak − yörünge; her kare için $M_t=\begin{bmatrix}\cos a&-\sin a&dx\\\sin a&\cos a&dy\end{bmatrix}$ (düzeltme), **tüm kareler tek** `F.affine_grid` + `grid_sample` çağrısıyla (`M`'yi normalize koordinatlara çevirip tersini al). `out (T, H, W)` `float32`.

In [ ]:
%%writefile stage_e.py
import math
import numpy as np
import cv2
import torch
import torch.nn.functional as F


def estimate_motion(frames):
    ...


def smooth(traj, r=5):
    ...


def stabilize(frames, r=5):
    ...


print("stabilizasyon")

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Zamansal gürültü giderme: düz ortalama, hareket kapılı ortalama

- `temporal_mean(frames, k)` — her kare için merkezi `2k+1` karenin piksel ortalaması (uçlarda mevcut olanlar).
- `gated_mean(frames, k, sigma, c=3.0)` — yalnız $|f_j-f_t|\le c\sigma$ olan karelerin ortalaması (merkez kare her zaman dahil): hareketli piksellerde hayaleti önler.
- `spatial(frames, s=1.5)` — her kareye `cv2.GaussianBlur(…, (0, 0), s)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def temporal_mean(frames, k):
    ...


def gated_mean(frames, k, sigma, c=3.0):
    ...


def spatial(frames, s=1.5):
    ...


print("zamansal")

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Horn–Schunck: küresel düzgünlükle optik akış

`horn_schunck(I0, I1, alpha=10.0, iters=200)` → `(u, v)`: $I_x, I_y, I_t$ Horn–Schunck'ın 2 × 2 × 2 küp ortalamalı farklarıyla (ya da merkezi fark; aşağıdaki kontrol yalnız doğruluğa bakar); Jacobi güncellemesi $u\leftarrow\bar u-I_x\frac{I_x\bar u+I_y\bar v+I_t}{\alpha^2+I_x^2+I_y^2}$, $\bar u$ komşu ortalaması (çekirdek $\frac1{12}\begin{bmatrix}1&2&1\\2&0&2\\1&2&1\end{bmatrix}$). Döngü yalnız iterasyonlarda.

In [ ]:
%%writefile extra_x2.py
import numpy as np
from scipy import ndimage


def horn_schunck(I0, I1, alpha=10.0, iters=200):
    ...


print("Horn–Schunck")

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Blok tabanlı FRUC ve örtüşen blok telafisi

Blok tabanlı FRUC: $I_0$, $I_2$'den $I_1$.

- `fruc_repeat(I0, I2)`, `fruc_average(I0, I2)`.
- `fruc_block(I0, I2, bs=8, R=8)` — $I_2$'nin blokları için $I_0$'da blok eşleme (vektör $d$), ara karede blok konumu $p+\frac d2$'ye yerleştirilecek değer $\frac12[I_0(p+d)+I_2(p)]$; bunu **ara kare ızgarasında** yapmak için simetrik arama kullan: ara karedeki her blok $q$ için $\min_d\text{SAD}(I_0(q+d), I_2(q-d))$, değer iki bloğun ortalaması (sınırları aşan adaylar elenir).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def fruc_repeat(I0, I2):
    ...


def fruc_average(I0, I2):
    ...


def fruc_block(I0, I2, bs=8, R=8):
    ...


print("FRUC")

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: RIFE: gerçek zamanlı ara akış kestirimiyle kare enterpolasyonu

[Huang, Zhang, Heng, Shi, Zhou — Real-Time Intermediate Flow Estimation for Video Frame Interpolation (ECCV 2022)](https://arxiv.org/abs/2011.06294)

d aşamasındaki "akış kestir → yarıya böl → çarpıt → harmanla" hattını, ara akışları ($F_{t\to0}$, $F_{t\to1}$) ve birleştirme maskesini doğrudan tahmin eden kaba-ince IFNet ile değiştir; ayrıcalıklı öğretmen (privileged distillation: eğitimde ara kareyi gören öğretmen blok) ile eğit. Vimeo-90K triplet (ya da kendi kısa video kliplerin) üzerinde PSNR/SSIM (M9.13) karşılaştır.


**Adımlar**

1. IFNet: 3 ölçekli blok, her biri önceki akışı ve çarpıtılmış kareleri alıp akış artığı + maske üretir.
2. Kayıplar: yeniden kurma (Laplacian piramidi L1), öğretmen damıtma, akış damıtma.
3. Taban çizgileri: kare tekrarı, ortalama, d aşamasındaki LK tabanlı MC-FRUC.
4. Keyfi $t$ için zaman girdisi; 720p'de süre ölçümü.

**Kabul ölçütleri**

- Vimeo-90K test (ya da kendi kümen) PSNR tablosu: RIFE > LK-MC > ortalama.
- Öğretmen damıtmanın kapalı/açık ablasyonu.
- Hızlı hareket ve örtme içeren örneklerde görsel karşılaştırma.

## Makale ödevi: RAFT: derin optik akış (uzantı)

[Teed & Deng — RAFT: Recurrent All-Pairs Field Transforms for Optical Flow (ECCV 2020)](https://arxiv.org/abs/2003.12039)

Tüm piksel çiftleri için 4-B korelasyon hacmi, çok ölçekli korelasyon havuzlama ve akışı GRU ile yinelemeli iyileştiren RAFT'ın küçük sürümünü (RAFT-S) uygula; FlyingChairs'te eğit, Sintel'de bu modüldeki piramitli LK ve Horn–Schunck ile uç nokta hatasını (EPE) karşılaştır.


**Adımlar**

1. Öznitelik ve bağlam kodlayıcıları, 1/8 çözünürlük.
2. Korelasyon hacmi $C=\langle g(I_0), g(I_1)\rangle$ ve 4 seviyeli havuzlama, yarıçap 3 bakış.
3. ConvGRU güncelleme, dışbükey üst örnekleme (convex upsampling).

**Kabul ölçütleri**

- Sintel clean EPE: RAFT-S < piramitli LK; iterasyon sayısına karşı EPE grafiği.

## Kaynaklar

- [Lucas & Kanade — An Iterative Image Registration Technique with an Application to Stereo Vision (IJCAI 1981)](https://www.ijcai.org/Proceedings/81-2/Papers/017.pdf) — ©
- [Bouguet — Pyramidal Implementation of the Lucas Kanade Feature Tracker (Intel, 2001)](https://web.stanford.edu/class/cs231m/references/pyr-lucas-kanade-feature-tracker-bouget.pdf) — ©
- [Horn & Schunck — Determining Optical Flow (AI 1981)](https://doi.org/10.1016/0004-3702(81)90024-2) — ©
- [Tekalp — Digital Video Processing, 2. baskı (blok eşleme, hareket telafisi)](https://www.pearson.com/) — ©
- [OpenCV — calcOpticalFlowPyrLK, estimateAffinePartial2D](https://docs.opencv.org/4.x/dc/d6b/group__video__track.html) — Apache-2.0